In [1]:
import os
os.chdir('/home/dominik/Downloads')

In [34]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import scipy.spatial as spatial

class ElasticBubbleCell:
    def __init__(self, position, radius=0.02):
        self.position = np.array(position, dtype=float)
        self.original_position = np.array(position, dtype=float)  # Reference position
        self.velocity = np.zeros(2, dtype=float)
        self.radius = radius
        self.is_t_cell = False
        
        # More nuanced interaction parameters
        self.t_cell_influence_radius = 0.2
        self.influence_strength = 0.4
        self.position_restoration_factor = 0.05
        self.randomness_factor = 0.001

    def interact(self, all_cells):
        # Find nearby T cells
        nearby_t_cells = [
            cell for cell in all_cells 
            if hasattr(cell, 'is_t_cell') and cell.is_t_cell and 
            np.linalg.norm(cell.nucleus - self.position) < self.t_cell_influence_radius
        ]
        
        # Base velocity computation
        if nearby_t_cells:
            # Weighted influence from T cell movement
            influences = []
            for cell in nearby_t_cells:
                # Distance-based weighting
                dist = np.linalg.norm(cell.nucleus - self.position)
                weight = 1 - (dist / self.t_cell_influence_radius)
                
                # Compute weighted velocity influence
                influence = cell.nucleus_velocity * weight * self.influence_strength
                influences.append(influence)
            
            # Average influences
            if influences:
                self.velocity = np.mean(influences, axis=0)
        else:
            # Minimal random movement and position restoration
            self.velocity = (
                np.random.normal(0, self.randomness_factor, 2) +
                (self.original_position - self.position) * self.position_restoration_factor
            )
        
        # Update position
        self.position += self.velocity
        
        # Soft boundary conditions
        self.position = np.clip(self.position, 0, 1)

class TCell:
    def __init__(self, position):
        # Multi-point representation
        self.nucleus = np.array(position, dtype=float)
        self.leading_edge = self.nucleus + np.random.uniform(-0.03, 0.03, 2)
        self.trailing_edge = self.nucleus - np.random.uniform(-0.03, 0.03, 2)
        
        # Movement parameters
        self.persistence = 0.95
        self.probing_intensity = 0.04
        self.exploration_factor = 0.02
        
        # Add velocity initialization
        self.leading_velocity = np.zeros(2)  # Explicitly initialize leading_velocity
        self.nucleus_velocity = np.zeros(2)
        self.trailing_velocity = np.zeros(2)
    
    def probe_environment(self, bubble_cells):
        """Simulate constrained probing in dense environment"""
        probing_force = np.zeros(2)
        deformation_accumulator = 0
        nearby_threshold = 0.02  # Tight interaction radius
        
        for cell in bubble_cells:
            dist = np.linalg.norm(self.leading_edge - cell.position)
            
            if dist < nearby_threshold:
                # Directional interaction
                direction = (self.leading_edge - cell.position) / (dist + 1e-5)
                
                # Progressive force based on proximity
                force_magnitude = (1 - dist / nearby_threshold) * self.probing_intensity
                probing_force += direction * force_magnitude
                
                # Deformation tracking
                deformation_accumulator += force_magnitude
        
        return probing_force, deformation_accumulator
    
    def move(self, bubble_cells):
        # Probing and force computation
        probing_force, deformation = self.probe_environment(bubble_cells)
        
        # Adaptive exploration with density influence
        exploration = np.random.uniform(-self.exploration_factor, self.exploration_factor, 2)
        
        # Leading edge movement
        self.leading_velocity = (
            self.leading_velocity * self.persistence + 
            probing_force + 
            exploration
        )
        
        # Velocity management
        speed = np.linalg.norm(self.leading_velocity)
        max_speed = 0.015
        if speed > max_speed:
            self.leading_velocity *= max_speed / speed
        
        # Update leading edge
        self.leading_edge += self.leading_velocity
        
        # Nucleus-leading edge dynamics
        nucleus_to_lead_vec = self.leading_edge - self.nucleus
        self.nucleus_velocity = nucleus_to_lead_vec * 0.5
        self.nucleus += self.nucleus_velocity
        
        # Trailing edge update
        self.trailing_velocity = (self.nucleus - self.trailing_edge) * 0.4
        self.trailing_edge += self.trailing_velocity
        
        # Boundary conditions
        for point in [self.nucleus, self.leading_edge, self.trailing_edge]:
            point[:] = np.clip(point, 0, 1)

def create_packed_cell_distribution(
    total_cells=400, 
    packing_density=0.7
):
    """Create a densely packed cell distribution"""
    cells = []
    attempts = 0
    max_attempts = 10000
    
    while len(cells) < total_cells and attempts < max_attempts:
        # Generate candidate position
        candidate = np.random.uniform(0, 1, 2)
        
        # Check overlap with existing cells
        is_valid = True
        for cell in cells:
            dist = np.linalg.norm(candidate - cell.position)
            if dist < packing_density:  # Tight packing radius
                is_valid = False
                break
        
        if is_valid:
            cells.append(ElasticBubbleCell(candidate))
        
        attempts += 1
    
    return cells

def simulate_lymph_node(
    t_cells_count=100, 
    bubble_cells_count=600,
    packing_density=0.7
):
    filename = f'tcell_simulation-t{t_cells_count}-b{bubble_cells_count}-d{packing_density}.mp4'
    
    # Create densely packed bubble cells
    bubble_cells = create_packed_cell_distribution(bubble_cells_count, packing_density)
    
    # Place T cells within this dense environment
    t_cells = [
        TCell(bubble_cells[np.random.randint(len(bubble_cells))].position + 
              np.random.uniform(-0.02, 0.02, 2)) 
        for _ in range(t_cells_count)
    ]
    
    # Visualization setup
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('black')
    ax.set_title('T Cell Migration through Dense Cellular Environment', color='white')

    # Scatter plot initialization
    bubble_scatter = ax.scatter([], [], c='lightblue', s=30, alpha=0.3)
    t_cell_nucleus = ax.scatter([], [], c='red', s=50, alpha=0.7)
    t_cell_leading = ax.scatter([], [], c='white', s=30, alpha=0.5)
    t_cell_trailing = ax.scatter([], [], c='blue', s=20, alpha=0.5)
    
    def update(frame):
        # Update bubble cell interactions to maintain density
        for cell in bubble_cells:
            cell.interact(bubble_cells)
        
        # Update T cell movements
        for t_cell in t_cells:
            t_cell.move(bubble_cells)
        
        # Position extraction
        bubble_pos = np.array([cell.position for cell in bubble_cells])
        t_cell_nucleus_pos = np.array([cell.nucleus for cell in t_cells])
        t_cell_leading_pos = np.array([cell.leading_edge for cell in t_cells])
        t_cell_trailing_pos = np.array([cell.trailing_edge for cell in t_cells])
        
        # Update scatter plots
        bubble_scatter.set_offsets(bubble_pos)
        t_cell_nucleus.set_offsets(t_cell_nucleus_pos)
        t_cell_leading.set_offsets(t_cell_leading_pos)
        t_cell_trailing.set_offsets(t_cell_trailing_pos)
        
        return [bubble_scatter, t_cell_nucleus, t_cell_leading, t_cell_trailing]

    # Animation creation
    ani = FuncAnimation(fig, update, frames=300, interval=50, blit=True)
    ani.save(filename, fps=20, dpi=150, writer='ffmpeg')
    plt.close(fig)

# Run simulation
for t in [50, 200]:
    for b in [200, 800]:
        for d in [0.001, 0.05]:
            simulate_lymph_node(t, b, d)


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

class TCell:
    def __init__(self, position):
        # Overall cell position (average of points)
        self.position = np.array(position, dtype=float)
        
        # Primary position is the nucleus
        self.nucleus = np.array(position, dtype=float)
        
        # Leading and trailing points with more dynamic potential
        self.leading_edge = self.nucleus + np.random.uniform(-0.05, 0.05, 2)
        self.trailing_edge = self.nucleus - np.random.uniform(-0.05, 0.05, 2)
        
        # Movement parameters for dynamic deformation
        self.nucleus_velocity = np.zeros(2)
        self.leading_velocity = np.zeros(2)
        self.trailing_velocity = np.zeros(2)
        
        # Elasticity and tension parameters
        self.rest_length = 0.05  # Default separation between points
        self.elasticity = 0.2    # How strongly points try to maintain rest length
        self.exploration_factor = 0.1  # Randomness in point movement
        
        # Directional persistence
        self.persistence = 0.7
        
        # Flag to distinguish T cells
        self.is_t_cell = True

    def compute_internal_forces(self):
        """Compute forces between nucleus, leading, and trailing points"""
        # Vector between leading edge and nucleus
        lead_nucleus_vec = self.leading_edge - self.nucleus
        lead_nucleus_dist = np.linalg.norm(lead_nucleus_vec)
        
        # Vector between nucleus and trailing edge
        trail_nucleus_vec = self.trailing_edge - self.nucleus
        trail_nucleus_dist = np.linalg.norm(trail_nucleus_vec)
        
        # Elastic forces to maintain rest length
        lead_force = lead_nucleus_vec * (lead_nucleus_dist - self.rest_length) * self.elasticity
        trail_force = trail_nucleus_vec * (trail_nucleus_dist - self.rest_length) * self.elasticity
        
        return lead_force, trail_force

    def find_nearest_target(self, all_cells):
        potential_targets = [
            other for other in all_cells 
            if not hasattr(other, 'is_t_cell') or not other.is_t_cell
        ]
        
        if potential_targets:
            # Calculate distances to all potential targets
            distances = [np.linalg.norm(self.position - other.position) for other in potential_targets]
            
            # Find the closest target
            closest_target_index = np.argmin(distances)
            return potential_targets[closest_target_index]
        
        return None

    def move(self, all_cells):
        # Find target for directional movement
        target = self.find_nearest_target(all_cells)
        
        # Compute internal elastic forces
        lead_force, trail_force = self.compute_internal_forces()
        
        # Exploration and target-seeking for leading edge
        if target is not None:
            # Vector towards target, biased towards leading edge
            target_vector = (target.position - self.leading_edge)
            leading_steering = target_vector * 0.02
        else:
            leading_steering = np.zeros(2)
        
        # Add some randomness to exploration
        exploration = np.random.uniform(-self.exploration_factor, self.exploration_factor, 2)
        
        # Update leading edge velocity
        self.leading_velocity = (
            self.leading_velocity * self.persistence + 
            leading_steering + 
            exploration + 
            lead_force * 2
        )
        
        # Limit leading edge velocity
        lead_speed = np.linalg.norm(self.leading_velocity)
        max_lead_speed = 0.03
        if lead_speed > max_lead_speed:
            self.leading_velocity *= max_lead_speed / lead_speed
        
        # Update leading edge position
        self.leading_edge += self.leading_velocity
        
        # Nucleus follows leading edge with some delay
        nucleus_to_lead_vec = self.leading_edge - self.nucleus
        self.nucleus_velocity = nucleus_to_lead_vec * 0.5
        self.nucleus += self.nucleus_velocity
        
        # Trailing edge pulled along, but with more resistance
        self.trailing_velocity = (self.nucleus - self.trailing_edge) * 0.3
        self.trailing_edge += self.trailing_velocity
        
        # Add some random exploration to trailing edge
        self.trailing_edge += np.random.uniform(-0.005, 0.005, 2)
        
        # Update overall cell position
        self.position = (self.nucleus + self.leading_edge + self.trailing_edge) / 3
        
        # Boundary conditions
        for point in [self.position, self.nucleus, self.leading_edge, self.trailing_edge]:
            point[:] = np.clip(point, 0, 1)

class OtherCell:
    def __init__(self, position):
        self.position = np.array(position, dtype=float)
        self.velocity = np.zeros(2, dtype=float)
        self.is_t_cell = False

    def move(self, all_cells):
        # Find nearby T cells
        nearby_t_cells = [
            cell for cell in all_cells 
            if hasattr(cell, 'is_t_cell') and cell.is_t_cell and np.linalg.norm(cell.nucleus - self.position) < 0.2
        ]
        
        if nearby_t_cells:
            # Weak influence from T cell movement
            influence = np.mean([cell.nucleus_velocity for cell in nearby_t_cells], axis=0)
            self.velocity = influence * 0.6
        else:
            # Very minimal random movement when no T cells nearby
            self.velocity = np.random.normal(0, 0.0005, 2)
        
        # Update position
        self.position += self.velocity
        
        # Boundary conditions
        self.position = np.clip(self.position, 0, 1)

def simulate_lymph_node(t_cells_count, other_cells_count, filename):
    # Create mixed cell population
    cells = (
        [TCell(np.random.uniform(0, 1, 2)) for _ in range(t_cells_count)] +
        [OtherCell(np.random.uniform(0, 1, 2)) for _ in range(other_cells_count)]
    )
    
    # Visualization setup
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('black')
    ax.set_title('T Cell Exploration Simulation', color='white')

    # Scatter plots for different cell components
    t_cell_nucleus = ax.scatter([], [], c='white', s=50, alpha=0.7)
    t_cell_leading = ax.scatter([], [], c='red', s=30, alpha=0.5)
    t_cell_trailing = ax.scatter([], [], c='blue', s=30, alpha=0.5)
    other_cells = ax.scatter([], [], c='lightblue', s=30, alpha=0.5)
    
    def update(frame):
        # Move all cells
        for cell in cells:
            cell.move(cells)
        
        # Extract positions
        t_cell_nucleus_pos = np.array([cell.nucleus for cell in cells if isinstance(cell, TCell)])
        t_cell_leading_pos = np.array([cell.leading_edge for cell in cells if isinstance(cell, TCell)])
        t_cell_trailing_pos = np.array([cell.trailing_edge for cell in cells if isinstance(cell, TCell)])
        other_cell_pos = np.array([cell.position for cell in cells if isinstance(cell, OtherCell)])
        
        # Update scatter plots
        t_cell_nucleus.set_offsets(t_cell_nucleus_pos)
        t_cell_leading.set_offsets(t_cell_leading_pos)
        t_cell_trailing.set_offsets(t_cell_trailing_pos)
        other_cells.set_offsets(other_cell_pos)
        
        return [t_cell_nucleus, t_cell_leading, t_cell_trailing, other_cells]

    # Create and save animation
    ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
    ani.save(filename, fps=15, dpi=100, writer='ffmpeg')
    plt.close(fig)

# Simulate different cell densities
DENSITIES = [(50, 100), (50, 500), (50, 1000)]
for i, (t_cells, other_cells) in enumerate(DENSITIES):
    simulate_lymph_node(t_cells, other_cells, filename=f't_cell_simulation_density_{i+1}.mp4')



In [48]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Define the parameters for the simulation
t_cells = 200
other_cells = 300
density = 0.1

class Cell:
    def __init__(self, position, is_t_cell=False):
        self.position = np.array(position, dtype=float)
        self.velocity = np.zeros(2, dtype=float)
        self.is_t_cell = is_t_cell
        
        # New attributes for leading/trailing edge dynamics
        self.polarization = np.zeros(2)  # Cell's polarization vector
        self.persistence = 0.7  # Tendency to maintain current direction
        self.exploration_factor = 0.2  # Randomness in movement

    def find_nearest_target(self, all_cells):
        # More sophisticated target selection
        potential_targets = [
            other for other in all_cells 
            if not other.is_t_cell and np.linalg.norm(self.position - other.position) < 0.5
        ]
        
        if potential_targets:
            # Weight targets by proximity and some randomness
            targets_with_weights = [
                (other, 1 / (np.linalg.norm(self.position - other.position) + 0.01))
                for other in potential_targets
            ]
            
            # Probabilistic target selection
            total_weight = sum(weight for _, weight in targets_with_weights)
            selection_prob = [weight / total_weight for _, weight in targets_with_weights]
            
            return np.random.choice(potential_targets, p=selection_prob)
        
        return None

    
    def move(self, all_cells):
        if self.is_t_cell:
            # Enhanced exploratory movement
            # 1. Persistent direction maintenance
            self.velocity = self.velocity * self.persistence
    
            # 2. Random exploration
            exploration_vector = np.random.uniform(-self.exploration_factor, self.exploration_factor, 2)
            self.velocity += exploration_vector
    
            # 3. Target-seeking with more biological realism
            target = self.find_nearest_target(all_cells)
            if target is not None:
                # Gradual steering instead of direct targeting
                desired_dir = (target.position - self.position)
                steering_vector = desired_dir * 0.01  # Subtle steering
                self.velocity += steering_vector
    
        # ----- Interaction force (unchanged) -----
        interaction_force = np.zeros(2)
        for other in all_cells:
            if other is self:
                continue
            
            distance = np.linalg.norm(self.position - other.position)
            
            # Wider, softer interaction zone
            if distance < 0.1:
                direction = (other.position - self.position) / (distance + 1e-5)
                
                # Very gentle repulsion that allows passing
                if distance < 0.05:
                    interaction_strength = -0.002 / (distance + 1e-5)
                else:
                    # Minimal attraction to prevent rigid clustering
                    interaction_strength = 0.0001 * np.exp(-distance/0.1)
                
                interaction_force += direction * interaction_strength

    
        # Integrate interaction force more subtly
        self.velocity += interaction_force * 0.5

        # For non-T cells, make movement more dependent on T cells
        if not self.is_t_cell:
            # Find nearby T cells
            nearby_t_cells = [
                cell for cell in all_cells 
                if cell.is_t_cell and np.linalg.norm(cell.position - self.position) < 0.2
            ]
            
            if nearby_t_cells:
                # Weak influence from T cell movement
                influence = np.mean([cell.velocity for cell in nearby_t_cells], axis=0)
                self.velocity = influence * 0.6
            else:
                # Very minimal random movement when no T cells nearby
                self.velocity = np.random.normal(0, 0.0005, 2)


        # ----- T‑cell directed movement -----
        if self.is_t_cell:
            # Find the closest non‑T‑cell (the “target”)
            target = None
            min_dist = np.inf
            for other in all_cells:
                if other.is_t_cell:
                    continue
                d = np.linalg.norm(self.position - other.position)
                if d < min_dist:
                    min_dist = d
                    target = other

            if target is not None:
                # Desired direction toward the target
                desired_dir = (target.position - self.position) / (min_dist + 1e-5)

                # Steering: gradually turn the current velocity toward the desired direction
                steering_strength = 0.03          # how quickly the cell can turn
                self.velocity += desired_dir * steering_strength

        # ----- Add interaction forces (optional) -----
        self.velocity += interaction_force

        # ----- Random diffusion (smaller amplitude) -----
        self.position += np.random.uniform(-0.0005, 0.0005, 2)

        # ----- Speed limiting -----
        speed = np.linalg.norm(self.velocity)
        max_speed = 0.02 if self.is_t_cell else 0.001   # a bit faster for T‑cells
        if speed > max_speed:
            self.velocity = self.velocity * (max_speed / speed)

        # Update position using the velocity vector
        self.position += self.velocity

        # ----- Boundary conditions -----
        self.position = np.clip(self.position, 0, 1)

class ExtracellularEnvironment:
    def __init__(self):
        pass

def simulate_lymph_node(t_cells, other_cells, density, filename):
    cells = [
        *[Cell(np.random.uniform(0, 1, 2), is_t_cell=True) for _ in range(t_cells)],
        *[Cell(np.random.uniform(0, 1, 2)) for _ in range(other_cells)]
    ]

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_facecolor('black')

    t_scatter = ax.scatter([], [], c='white', s=50)
    other_scatter = ax.scatter([], [], c='lightblue', s=30)

    def update(frame):
        for cell in cells:
            cell.move(cells)
        
        positions = np.array([c.position for c in cells])
        
        t_positions = [cell.position for cell in cells if cell.is_t_cell]
        other_positions = [cell.position for cell in cells if not cell.is_t_cell]
        
        t_scatter.set_offsets(t_positions)
        other_scatter.set_offsets(other_positions)
        
        return [t_scatter, other_scatter]

    ani = FuncAnimation(fig, update, frames=200, interval=50, blit=True)
    ani.save(filename, fps=15, dpi=100)
    plt.close(fig)

# Simulate different cell densities
DENSITIES = [(50, 100), (50, 500), (50, 1000)]
for i, (t_cells, other_cells) in enumerate(DENSITIES):
    simulate_lymph_node(t_cells, other_cells, density=DENSITIES[i], filename=f't_cell_simulation_density_{i+1}.mp4')